## RLVR Example - Finetuning with Sagemaker

This notebook demonstrates basic user flow for RLVR Finetuning from a model available in Sagemaker Jumpstart.
Information on available models on jumpstart: https://docs.aws.amazon.com/sagemaker/latest/dg/jumpstart-foundation-models-latest.html

### Setup and Configuration

Initialize the environment by importing necessary libraries and configuring AWS credentials

In [ ]:
# Configure AWS credentials and region
#! ada credentials update --provider=isengard --account=<> --role=Admin --profile=default --once
#! aws configure set region us-west-2

from sagemaker.train.rlvr_trainer import RLVRTrainer
from sagemaker.train.configs import InputData
from rich import print as rprint
from rich.pretty import pprint
from sagemaker.core.resources import ModelPackage
from sagemaker.train.common import TrainingType


import boto3
import os
from sagemaker.core.helper.session_helper import Session

# For MLFlow native metrics in Trainer wait, run below line with approriate region
os.environ["SAGEMAKER_MLFLOW_CUSTOM_ENDPOINT"] = "https://mlflow.sagemaker.us-west-2.app.aws"



### Prepare and Register Dataset
Prepare and Register Dataset for Finetuning

In [ ]:
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.dataset_utils import CustomizationTechnique



# Register dataset in SageMaker AI Registry
# This creates a versioned dataset that can be referenced by ARN
# Provide a source (it can be local file path or S3 URL)
dataset = DataSet.create(
    name="demo-2",
    source="s3://mc-flows-sdk-testing/input_data/rlvr-rlaif-test-data/train_285.jsonl"
)

print(f"Dataset ARN: {dataset.arn}")

##### Create a Model Package group (if not already exists)

In [ ]:
from sagemaker.core.resources import ModelPackage, ModelPackageGroup

model_package_group=ModelPackageGroup.create(model_package_group_name="test-model-package-group")

### Create RLVRTrainer
**Required Parameters** 

* `model`: base_model id on Sagemaker Hubcontent that is available to finetune (or) ModelPackage artifacts

**Optional Parameters**
* `custom_reward_function`: Custom reward function/Evaluator ARN
* `model_package_group`: ModelPackage group name or ModelPackageGroup object. This parameter is mandatory when a base model ID is provided, but optional when a model package is provided.
* `mlflow_resource_arn`: MLFlow app ARN to track the training job
* `mlflow_experiment_name`: MLFlow app experiment name(str)
* `mlflow_run_name`: MLFlow app run name(str)
* `training_dataset`: Training Dataset - should be a Dataset ARN or Dataset object (Please note training dataset is required for a training job to run, can be either provided via Trainer or .train())
* `validation_dataset`: Validation Dataset - should be a Dataset ARN or Dataset object
* `s3_output_path`: S3 path for the trained model artifacts

#### Reference 
Refer this doc for other models that support Model Customization: 
https://docs.aws.amazon.com/bedrock/latest/userguide/custom-model-supported.html

In [ ]:
# For fine-tuning (prod)
rlvr_trainer = RLVRTrainer(
    model="meta-textgeneration-llama-3-2-1b-instruct", 
    model_package_group=model_package_group, # or use an existing model package group arn
    mlflow_experiment_name="test-rlvr-finetuned-models-exp", 
    mlflow_run_name="test-rlvr-finetuned-models-run", 
    training_dataset=dataset.arn,
    s3_output_path="s3://mc-flows-sdk-testing/output/",
    accept_eula=True
)

### Discover and update Finetuning options

Each of the technique and model has overridable hyperparameters that can be finetuned by the user.

In [ ]:
print("Default Finetuning Options:")
pprint(rlvr_trainer.hyperparameters.to_dict()) # rename as hyperparameters

#set options
rlvr_trainer.hyperparameters.get_info()



#### Start RLVR training


### Dry Run (Validation Only)

Use `dry_run=True` to validate IAM permissions, data paths, hyperparameters, infrastructure without submitting a job.

In [ ]:
# Validate configuration without launching a training job
rlvr_trainer.train(dry_run=True)
print("Dry-run passed — configuration is valid.")

In [ ]:
training_job = rlvr_trainer.train(wait=True)

In [ ]:
import re
from sagemaker.core.utils.utils import Unassigned
import json


def pretty_print(obj):
    def parse_unassigned(item):
        if isinstance(item, Unassigned):
            return None
        if isinstance(item, dict):
            return {k: parse_unassigned(v) for k, v in item.items() if parse_unassigned(v) is not None}
        if isinstance(item, list):
            return [parse_unassigned(x) for x in item if parse_unassigned(x) is not None]
        if isinstance(item, str) and "Unassigned object" in item:
            pairs = re.findall(r"(\w+)=([^<][^=]*?)(?=\s+\w+=|$)", item)
            result = {k: v.strip("'\"") for k, v in pairs}
            return result if result else None
        return item

    cleaned = parse_unassigned(obj.__dict__ if hasattr(obj, '__dict__') else obj)
    print(json.dumps(cleaned, indent=2, default=str))

# Usage
pretty_print(training_job)

In [ ]:
training_job = rlvr_trainer.train(wait=True)

### View any Training job details

We can get any training job details and its status with TrainingJob.get(...)

In [ ]:
from sagemaker.core.resources import TrainingJob

response = TrainingJob.get(training_job_name="meta-textgeneration-llama-3-2-3b-instruct-rlvr-20251123033517")
pretty_print(response)

In [ ]:
training_job.refresh()
pretty_print(training_job)

### Test RLVR with Custom RewardFunction

Here we are providing a user-defined reward function ARN

#### Option 1: Create an Evaluator manually and pass it to the Trainer

In [ ]:
from rich.pretty import pprint

from sagemaker.ai_registry.air_constants import REWARD_FUNCTION, REWARD_PROMPT
from sagemaker.ai_registry.evaluator import Evaluator

# Method : Lambda
evaluator = Evaluator.create(
    name = "sdk-new-rf11",
    source="arn:aws:lambda:us-west-2:<>:function:sm-eval-vinayshm-rlvr-llama-321b-instruct-v1-<>8",
    type=REWARD_FUNCTION

)

#### Verify the reward function before training

Before kicking off a (potentially long-running) training job, the reward function can be validated that it returns output in the expected format. `verify_reward_function` runs your reward function against a few sample records and checks that each result contains an `id` and a numeric `aggregate_reward_score`.

It works with either a **local Python file** (containing a `lambda_handler`) or a **deployed Lambda ARN**. For Nova models the `platform` parameter is required when passing a Lambda ARN.


In [ ]:
from sagemaker.train.common_utils.rlvr_reward_verifier import verify_reward_function

# A couple of sample records that mirror your training data format.
sample_data = [
    {
        "id": "sample_1",
        "messages": [
            {"role": "user", "content": "What is 2 + 2?"},
            {"role": "assistant", "content": "The answer is 4. #### 4"},
        ],
        "reference_answer": "4",
    },
    {
        "id": "sample_2",
        "messages": [
            {"role": "user", "content": "What is the capital of France?"},
            {"role": "assistant", "content": "Paris #### Paris"},
        ],
        "reference_answer": "Paris",
    },
]

# Option 1: verify a deployed Lambda reward function (the evaluator created above).
result = verify_reward_function(
    reward_function=evaluator.reference,  # Lambda ARN
    sample_data=sample_data,
    is_nova=False,  # set True for Nova models

)

# Option 2: verify a local reward function file before deploying it.
verify_reward_function(
    reward_function="./my_reward_fn.py",
    sample_data=sample_data,
    is_nova=False,
)

pprint(result)
assert result["success"], "Reward function verification failed - fix the output format before training."


#### Use it with RLVR Trainer

In [ ]:

# For fine-tuning 
rlvr_trainer = RLVRTrainer(
    model="meta-textgeneration-llama-3-2-1b-instruct", # Union[str, ModelPackage] 
    model_package_group="sdk-test-finetuned-models", # Make it Optional
    mlflow_experiment_name="test-rlvr-finetuned-models-exp", # Optional[str]
    mlflow_run_name="test-rlvr-finetuned-models-run", # Optional[str]
    training_dataset=dataset, #Optional[]
    s3_output_path="s3://mc-flows-sdk-testing/output/",
    custom_reward_function=evaluator,
    accept_eula=True
    
)

In [ ]:
training_job = rlvr_trainer.train(wait=True)

In [ ]:
#training_job.refresh()
pretty_print(training_job)

#### Option 2: Pass a Lambda ARN directly (auto-creates Evaluator)

You can pass an AWS Lambda function ARN directly to `custom_reward_function`. The SDK will automatically:
1. Derive an evaluator name from the Lambda function name
2. Check if an Evaluator with that name already exists and points to the same Lambda (reuses it if so)
3. Otherwise, create a new Evaluator in the AI Registry and use its ARN

This simplifies the workflow — no need to manually create an Evaluator object first.

In [ ]:
# Pass Lambda ARN directly - the SDK will create the Evaluator automatically
lambda_arn = "arn:aws:lambda:us-west-2:<account_id>:function:my-rlvr-reward-function"

rlvr_trainer = RLVRTrainer(
    model="meta-textgeneration-llama-3-2-1b-instruct",
    model_package_group="sdk-test-finetuned-models",
    mlflow_experiment_name="test-rlvr-finetuned-models-exp",
    mlflow_run_name="test-rlvr-finetuned-models-run",
    training_dataset=dataset,
    s3_output_path="s3://mc-flows-sdk-testing/output/",
    custom_reward_function=lambda_arn,  # Lambda ARN passed directly!
    accept_eula=True
)

In [ ]:
training_job = rlvr_trainer.train(wait=True)

In [ ]:
pretty_print(training_job)

## Continued Finetuning (or) Finetuning on Model Artifacts

#### Discover a ModelPackage and get its details

In [ ]:
from rich import print as rprint
from rich.pretty import pprint
from sagemaker.core.resources import ModelPackage, ModelPackageGroup

#model_package_iter = ModelPackage.get_all(model_package_group_name="test-finetuned-models-gamma")
model_package = ModelPackage.get(model_package_name="arn:aws:sagemaker:us-west-2:<>:model-package/test-finetuned-models-gamma/61")

pretty_print(model_package)

#### Create Trainer

Trainer creation is same as above Finetuning Section except for `model`'s input is ModelPackage(previously trained artifacts)

In [ ]:
# For fine-tuning 
rlvr_trainer = RLVRTrainer(
    model=model_package, # Union[str, ModelPackage] 
    training_type=TrainingType.LORA, 
    model_package_group="test-finetuned-models-gamma", #"test-finetuned-models", # Make it Optional
    mlflow_resource_arn="arn:aws:sagemaker:us-west-2:<>:mlflow-tracking-server/mmlu-eval-experiment",  # Optional[str] - MLflow app ARN (auto-resolved if not provided), can accept name and search in the account
    mlflow_experiment_name="test-rlvr-finetuned-models-exp", # Optional[str]
    mlflow_run_name="test-rlvr-finetuned-models-run", # Optional[str]
    training_dataset=dataset.arn,     #"arn:aws:sagemaker:us-west-2:<>:hub-content/AIRegistry/DataSet/MarketingDemoDataset1/1.0.0", #Optional[]
    s3_output_path="s3://open-models-testing-pdx/output",
    accept_eula=True
)

#### Start the Training

In [ ]:
training_job = rlvr_trainer.train(
    wait=True,
)

In [ ]:
pretty_print(training_job)

## Iterative Training (Resume from Checkpoint)

Resume RLVR training from a previously trained model package. Pass the model package directly — the service uses `SourceModelPackageArn` to resume from that checkpoint.

#### Fetch the Model Package ARN from a previous Training Job

After a serverless training job completes, it produces an `OutputModelPackageArn` that can be used as the base model for iterative (continued) fine-tuning. Here's how to retrieve it:

In [ ]:
from sagemaker.core.resources import TrainingJob

# Get the completed training job
previous_job = TrainingJob.get(training_job_name="<your-previous-training-job-name>")

# The output model package ARN is available on completed serverless training jobs
output_model_package_arn = previous_job.output_model_package_arn
print(f"Output Model Package ARN: {output_model_package_arn}")

# Use this ARN to get the ModelPackage for iterative training
from sagemaker.core.resources import ModelPackage

model_package = ModelPackage.get(model_package_name=output_model_package_arn)
pretty_print(model_package)

In [ ]:
# Iterative RLVR training from a model package
# Pass the model package from a previous training run
rlvr_trainer_iterative = RLVRTrainer(
    model=model_package,  # ModelPackage from a previous training run
    training_type=TrainingType.LORA,
    model_package_group="my-iterative-models",
    training_dataset=dataset.arn,
    s3_output_path="s3://my-bucket/iterative-output/",
    accept_eula=True,
)

training_job = rlvr_trainer_iterative.train(wait=False)
print(f"Iterative RLVR job submitted: {training_job.training_job_name}")

#### Nova RLVR job

In [ ]:
import os
os.environ['SAGEMAKER_REGION'] = 'us-east-1'

# For fine-tuning 
rlvr_trainer = RLVRTrainer(
    model="nova-textgeneration-lite-v2", # Union[str, ModelPackage] 
    model_package_group="sdk-test-finetuned-models", #"test-finetuned-models", # Make it Optional
    #mlflow_resource_arn="arn:aws:sagemaker:us-east-1:<>:mlflow-app/app-UNBKLOAX64PX",  # Optional[str] - MLflow app ARN (auto-resolved if not provided), can accept name and search in the account
    mlflow_experiment_name="test-nova-rlvr-finetuned-models-exp", # Optional[str]
    mlflow_run_name="test-nova-rlvr-finetuned-models-run", # Optional[str]
    training_dataset="s3://mc-flows-sdk-testing-us-east-1/input_data/rlvr-nova/grpo-64-sample.jsonl",
    validation_dataset="s3://mc-flows-sdk-testing-us-east-1/input_data/rlvr-nova/grpo-64-sample.jsonl",
    s3_output_path="s3://mc-flows-sdk-testing-us-east-1/output/",
    custom_reward_function="arn:aws:sagemaker:us-east-1:<>:hub-content/sdktest/JsonDoc/rlvr-nova-test-rf/0.0.1",
    accept_eula=True
)


In [ ]:
rlvr_trainer.hyperparameters.to_dict()

In [ ]:
rlvr_trainer.hyperparameters.data_s3_path = 's3://example-bucket'

rlvr_trainer.hyperparameters.reward_lambda_arn = 'arn:aws:lambda:us-east-1:<>:function:rlvr-nova-reward-function'

In [ ]:
rlvr_trainer.hyperparameters.to_dict()

In [ ]:
training_job = rlvr_trainer.train(wait=True)